# 01 - Data quality audit (Process A, then Process B)

Goal: a quick, repeatable look at the **calibration** and **test** data before any modelling:
missing values, class imbalance, batch lengths, sensor behaviour (constant / stuck / outliers),
correlation, calibration-vs-test shift, and a cleaning plan
(which columns and rows to drop, and why).

**Batches have different numbers of rows** (Process A: roughly 200 to 2900 rows; Process B: fixed length).
So nothing here is computed on the pooled rows alone:

- every statistic is computed **per batch first** and then aggregated across batches, or
- rows are **weighted by 1/n_rows** of their batch, so each batch counts once whatever its length.

Layout: shared helpers, then Section A, then Section B (identical steps), then a side-by-side summary.
Per-batch tables are saved to `outputs/` (gitignored).

In [ ]:
import hashlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

from brpmm.config import BATCH_COL, LABEL_ACCEPTED, LABEL_REJECTED, OUTPUTS_DIR
from brpmm.data import (
    CONSTANT_COLUMNS,
    TIME_COL,
    load_calibration,
    load_test,
    sensor_columns,
    to_binary,
)

pd.set_option("display.float_format", "{:.3f}".format)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

SEED = 0  # fixed seed (STRATEGY.md: all methods share the same splits)
N_SPLITS = 5  # STRATEGY.md: stratified 5-fold CV
COLORS = {LABEL_ACCEPTED: "#0072B2", LABEL_REJECTED: "#D55E00"}  # colour-blind safe
MISSING_DROP_THRESHOLD = (
    0.5  # drop a sensor if more than 50% of its values are missing (rule of thumb: 50-60%)
)
# Cleaning rules (steps X.9 and X.10)
LOW_CARDINALITY = 5  # numeric sensor with <= this many distinct values acts like a switch
NEAR_CONSTANT_SHARE = 0.95  # one single value in more than 95% of the rows
COLLINEAR_R = 0.95  # |r| above this: one sensor of the pair is a drop candidate
SPIKE_WINDOW, SPIKE_K = 11, 10  # spike = more than 10 robust SDs from the local rolling median
DROP_COLLINEAR = False  # True: also drop the collinear candidates (reasons in step X.9)
DROP_SPIKE_ROWS = False  # True: also drop rows with an extreme spike (reasons in step X.10)
summary = {}  # filled by each process section, compared at the end

## Helpers

Kept in the notebook so this stays a self-contained script. Once the team agrees on them, they belong in
`src/brpmm` (e.g. `features.py`), as the README convention asks.

In [ ]:
# --- loading and structure ------------------------------------------------------------------
def load(process):
    """Calibration series, labels and test series. Constant columns are kept on purpose:
    detecting them is part of the audit."""
    cal, labels = load_calibration(process, drop_constant=False)
    test = load_test(process, drop_constant=False)
    return cal, labels, test


def batch_lengths(df):
    """Number of rows (time steps) of each batch."""
    return df.groupby(BATCH_COL, sort=False).size().rename("n_rows")


def structure_checks(df, sensors):
    """Integrity checks on the long-format table."""
    n_batches = df[BATCH_COL].nunique()
    block_starts = df[BATCH_COL].ne(df[BATCH_COL].shift()).sum()
    numeric = df[sensors].select_dtypes("number")
    return pd.Series(
        {
            "rows": len(df),
            "batches": n_batches,
            "batch rows contiguous": bool(block_starts == n_batches),
            "non-numeric sensor columns": [c for c in sensors if c not in numeric.columns],
            "inf values": int(np.isinf(numeric).to_numpy().sum()),
            "fully duplicated rows": int(df.duplicated(subset=[BATCH_COL, *sensors]).sum()),
        }
    )


# --- missing values and runs ------------------------------------------------------------------
def missing_report(df, sensors):
    """Missing values per sensor, pooled AND batch-averaged (they differ when the
    missing cells sit in short or long batches)."""
    isna = df[sensors].isna()
    per_batch = isna.groupby(df[BATCH_COL], sort=False).mean()  # batch x sensor, fraction missing
    table = pd.DataFrame(
        {
            "pooled_%": isna.mean() * 100,
            "batch_avg_%": per_batch.mean() * 100,
            "batches_affected": (per_batch > 0).sum(),
            "worst_batch_%": per_batch.max() * 100,
        }
    )
    return table, per_batch


def high_missing_sensors(per_batch, threshold=MISSING_DROP_THRESHOLD):
    """Sensors to drop: batch-averaged missing fraction above `threshold`.

    Batch-averaged so each batch counts once whatever its length. Decide on the calibration
    data and apply the same list to the test data, so both keep identical columns."""
    frac = per_batch.mean()
    return frac[frac > threshold].rename("missing_fraction")


def _longest_true_run(mask):
    if not mask.any():
        return 0
    edges = np.diff(np.concatenate(([0], mask.astype(np.int8), [0])))
    return int((np.flatnonzero(edges == -1) - np.flatnonzero(edges == 1)).max())


def longest_run_fraction(df, sensors, kind):
    """Longest run per batch and sensor, as a FRACTION of the batch length
    (so batches of different length are comparable).
    kind="nan": consecutive missing values.
    kind="stuck": consecutive identical NON-ZERO values (a run of exact zeros is an actuator
    that is switched off, e.g. feed or acid flow, not a frozen sensor)."""
    rows = {}
    for bid, g in df.groupby(BATCH_COL, sort=False):
        x = g[sensors].to_numpy(dtype=float)
        if kind == "nan":
            runs = [_longest_true_run(np.isnan(x[:, j])) for j in range(x.shape[1])]
        else:
            same = (x[1:] == x[:-1]) & (x[1:] != 0)
            runs = [_longest_true_run(same[:, j]) + 1 for j in range(x.shape[1])]
        rows[bid] = np.array(runs) / len(g)
    return pd.DataFrame.from_dict(rows, orient="index", columns=sensors)

In [ ]:
# --- batch length, class imbalance, weights ---------------------------------------------------
def length_vs_label(lengths, labels):
    """Length summary per class + AUC of length as a predictor of 'rejected'
    (0.5 = unrelated, below 0.5 = rejected batches tend to be shorter)."""
    lab = labels.loc[lengths.index]
    return lengths.groupby(lab).describe(), roc_auc_score(to_binary(lab), lengths)


def flag_short_batches(lengths):
    """Boolean Series, True = suspiciously short batch (e.g. an aborted run).

    TODO(human): decide the rule. Options: a fraction of the median length
    (lengths < 0.5 * lengths.median()), a robust z-score on the length, or a fixed
    number of rows agreed with the mentors.
    """
    return pd.Series(False, index=lengths.index)  # placeholder: flags nothing yet


def imbalance_report(labels, lengths):
    """Class counts at batch level AND row level, plus the 'accept everything' F1
    (accepted = positive class, as on the leaderboard: precision = share, recall = 1)."""
    counts = labels.value_counts()
    row_share = lengths.groupby(labels.loc[lengths.index]).sum() / lengths.sum()
    table = pd.DataFrame(
        {
            "batches": counts,
            "batch_share_%": counts / counts.sum() * 100,
            "row_share_%": row_share * 100,
        }
    )
    share_acc = counts[LABEL_ACCEPTED] / counts.sum()
    return table, 2 * share_acc / (1 + share_acc)


def class_weights(labels):
    """'Balanced' class weights at batch level: n_batches / (n_classes * n_in_class)."""
    counts = labels.value_counts()
    return (len(labels) / (len(counts) * counts)).to_dict()


def row_weights(df, labels):
    """Sample weight per row. Each batch gets a TOTAL weight equal to its class weight,
    spread evenly over its rows. Handles class imbalance and unequal batch lengths at once."""
    per_batch = labels.map(class_weights(labels))
    n_rows = df.groupby(BATCH_COL, sort=False)[BATCH_COL].transform("size")
    return df[BATCH_COL].map(per_batch) / n_rows


def fold_report(labels, lengths, n_splits=N_SPLITS, seed=SEED):
    """Held-out composition of each stratified fold (rejected count, mean batch length)."""
    skf = StratifiedKFold(n_splits, shuffle=True, random_state=seed)
    rows = []
    for k, (_, held_idx) in enumerate(skf.split(labels.index, labels)):
        held = labels.iloc[held_idx]
        rows.append(
            {
                "fold": k,
                "batches": len(held),
                "rejected": int((held == LABEL_REJECTED).sum()),
                "mean_rows": lengths.loc[held.index].mean(),
            }
        )
    return pd.DataFrame(rows).set_index("fold")

In [ ]:
# --- sensor characteristics -------------------------------------------------------------------
def batch_stats(df, sensors):
    """Per-batch mean / std / min / max (each a batch x sensor table)."""
    g = df.groupby(BATCH_COL, sort=False)[sensors]
    return {"mean": g.mean(), "std": g.std(), "min": g.min(), "max": g.max()}


def sensor_overview(df, sensors, stats):
    """One row per sensor. between = spread of batch means, within = typical in-batch spread."""
    top_share = {
        c: df[c].value_counts(normalize=True).iloc[0] if df[c].notna().any() else np.nan
        for c in sensors
    }
    out = pd.DataFrame(
        {
            "n_unique": df[sensors].nunique(),
            "top_value_share_%": pd.Series(top_share) * 100,
            "zero_share_%": (df[sensors] == 0).mean() * 100,
            "min": df[sensors].min(),
            "max": df[sensors].max(),
            "between_batch_sd": stats["mean"].std(),
            "within_batch_sd": stats["std"].mean(),
            "n_negative": (df[sensors] < 0).sum(),
        }
    )
    out["between/within"] = out["between_batch_sd"] / out["within_batch_sd"]
    return out


def robust_z(x):
    med = x.median()
    mad = (x - med).abs().median().replace(0, np.nan)
    return (x - med) / (1.4826 * mad)


def batch_outlier_table(stats, threshold=4.0):
    """Batches whose per-batch mean or std is far from the other batches (robust z-score)."""
    z = pd.concat({k: robust_z(stats[k]) for k in ("mean", "std")}, axis=1).abs()
    worst = z.idxmax(axis=1)  # (statistic, sensor) with the largest |z| in each batch
    return pd.DataFrame(
        {
            "n_extreme_stats": (z > threshold).sum(axis=1),
            "max_abs_z": z.max(axis=1),
            "worst_stat": [w[0] for w in worst],
            "worst_sensor": [w[1] for w in worst],
        },
        index=z.index,
    )


def row_outlier_share(df, sensors, k=3.0):
    """Fraction of rows outside the pooled IQR fences, per batch and sensor.
    Rough guide only: for processes with strong dynamics many 'outliers' are real phases.
    Sensors whose IQR is 0 are returned as NaN."""
    q1, q3 = df[sensors].quantile(0.25), df[sensors].quantile(0.75)
    iqr = (q3 - q1).replace(0, np.nan)  # IQR = 0 (mostly-constant sensor): fences are meaningless
    out = ((df[sensors] < q1 - k * iqr) | (df[sensors] > q3 + k * iqr)).astype(float)
    out.loc[:, iqr.isna()] = np.nan
    return out.groupby(df[BATCH_COL], sort=False).mean()


def label_effect_sizes(stats, labels):
    """Cohen's d of the per-batch mean, rejected minus accepted (noisy when few rejected)."""
    m = stats["mean"].loc[labels.index]
    a, r = m[labels == LABEL_ACCEPTED], m[labels == LABEL_REJECTED]
    pooled = np.sqrt(((len(a) - 1) * a.var() + (len(r) - 1) * r.var()) / (len(a) + len(r) - 2))
    d = ((r.mean() - a.mean()) / pooled).rename("cohens_d").dropna()
    return d.sort_values(key=np.abs, ascending=False)


def weighted_corr(df, sensors):
    """Correlation matrix with rows weighted by 1/n_rows of their batch (each batch counts once)."""
    w = 1.0 / df.groupby(BATCH_COL, sort=False)[BATCH_COL].transform("size")
    ok = df[sensors].notna().all(axis=1)
    cov = np.cov(df.loc[ok, sensors].to_numpy().T, aweights=w[ok].to_numpy())
    sd = np.sqrt(np.diag(cov))
    return pd.DataFrame(cov / np.outer(sd, sd), index=sensors, columns=sensors)


def calibration_vs_test(cal_stats, test_stats):
    """Shift of the average batch-mean in test vs calibration, in calibration batch-to-batch SDs."""
    return (
        (test_stats["mean"].mean() - cal_stats["mean"].mean()) / cal_stats["mean"].std()
    ).rename("shift_in_cal_SDs")

In [ ]:
# --- plots ------------------------------------------------------------------------------------
def plot_length_hist(lengths, labels):
    fig, ax = plt.subplots(figsize=(7, 3))
    bins = np.histogram_bin_edges(lengths, bins=30)
    for lab in (LABEL_ACCEPTED, LABEL_REJECTED):
        ids = labels.index[labels == lab]
        ax.hist(
            lengths.loc[ids], bins=bins, alpha=0.65, color=COLORS[lab], label=f"{lab} ({len(ids)})"
        )
    ax.set(xlabel="rows per batch", ylabel="number of batches", title="Batch length by label")
    ax.legend()
    plt.show()


def plot_corr(corr):
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(corr, vmin=-1, vmax=1, cmap="RdBu_r")
    ax.set_xticks(range(len(corr)), corr.columns, rotation=90)
    ax.set_yticks(range(len(corr)), corr.index)
    fig.colorbar(im, ax=ax, label="weighted correlation")
    ax.set_title("Sensor correlation (each batch weighted equally)")
    plt.show()


def plot_trajectories(df, labels, sensors, n_per_class=6, ncols=4):
    """A few batches per class, with time rescaled to 0..1 so different lengths overlay."""
    rng = np.random.default_rng(SEED)
    picked = {
        lab: rng.choice(
            labels.index[labels == lab], min(n_per_class, (labels == lab).sum()), replace=False
        )
        for lab in (LABEL_ACCEPTED, LABEL_REJECTED)
    }
    nrows = int(np.ceil(len(sensors) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.6 * ncols, 2.4 * nrows), squeeze=False)
    groups = dict(tuple(df.groupby(BATCH_COL, sort=False)))
    for ax, s in zip(axes.ravel(), sensors, strict=False):
        for lab, ids in picked.items():
            for bid in ids:
                g = groups[bid]
                ax.plot(
                    g[TIME_COL] / max(len(g) - 1, 1), g[s], color=COLORS[lab], alpha=0.5, lw=0.8
                )
        ax.set_title(s, fontsize=9)
    for ax in axes.ravel()[len(sensors) :]:
        ax.axis("off")
    fig.suptitle("Trajectories (blue = accepted, orange = rejected; time rescaled to 0..1)")
    fig.tight_layout()
    plt.show()

In [ ]:
# --- cleaning decisions -----------------------------------------------------------------------
def cardinality_report(df, sensors):
    """Distinct values per sensor. Low cardinality = switch-like numeric sensor; high cardinality
    only matters for non-numeric (ID-like) columns; a continuous sensor has many values."""
    n_unique = df[sensors].nunique()
    numeric = df[sensors].dtypes.map(pd.api.types.is_numeric_dtype)
    return pd.DataFrame(
        {
            "n_unique": n_unique,
            "unique_ratio": n_unique / len(df),
            "low_cardinality": n_unique <= LOW_CARDINALITY,
            "high_cardinality": (~numeric) & (n_unique / len(df) > 0.5),
        }
    )


def low_variance_report(overview):
    """Zero-variance and near-constant sensors. Scale-free on purpose: a threshold on the raw
    variance would depend on the unit (a pH sensor and a flow in L/h cannot share one)."""
    return pd.DataFrame(
        {
            "zero_variance": overview["n_unique"] <= 1,
            "near_constant": (overview["top_value_share_%"] > NEAR_CONSTANT_SHARE * 100)
            & (overview["n_unique"] > 1),
        }
    )


def collinear_candidates(corr, threshold=COLLINEAR_R):
    """Greedy: while a pair has |r| > threshold, propose dropping the sensor of the pair that has
    the larger total |r| with the remaining sensors."""
    names = list(corr.columns)
    a = np.nan_to_num(corr.abs().to_numpy(copy=True))
    np.fill_diagonal(a, 0)
    alive = np.ones(len(names), dtype=bool)
    rows = []
    while True:
        sub = a * np.outer(alive, alive)
        i, j = np.unravel_index(np.argmax(sub), sub.shape)
        if sub[i, j] <= threshold:
            break
        drop, keep = (i, j) if sub[i].sum() >= sub[j].sum() else (j, i)
        alive[drop] = False
        rows.append((names[drop], names[keep], round(float(sub[i, j]), 3)))
    return pd.DataFrame(rows, columns=["drop", "because_of", "abs_r"])


def batch_fingerprints(df, sensors):
    """One hash per batch of its full sensor trajectory: finds duplicated batches, also across
    calibration and test (leakage)."""
    return df.groupby(BATCH_COL, sort=False)[sensors].apply(
        lambda g: hashlib.md5(g.to_numpy(dtype=float).tobytes()).hexdigest()
    )


def spike_mask(df, sensors, window=SPIKE_WINDOW, k=SPIKE_K):
    """True where a value is an extreme spike: more than k robust SDs away from the rolling median
    of its OWN batch. A batch that is merely shifted to another level is not flagged here (that is
    a batch-level anomaly, see the outlier-batch table). The robust SD is 1.4826 x MAD of the
    residuals, taken over all batches."""
    trend = df.groupby(BATCH_COL, sort=False)[sensors].transform(
        lambda s: s.rolling(window, center=True, min_periods=1).median()
    )
    resid = df[sensors] - trend
    mad = (resid - resid.median()).abs().median().replace(0, np.nan)
    return resid.abs() > k * 1.4826 * mad


def plot_boxplots(df, labels, sensors, stats):
    """Box plots, robust-scaled ((x - median) / IQR) so sensors with different units share one axis.
    Left: rows (sample with every batch weighted equally). Right: one dot per batch (its mean),
    rejected batches in orange."""
    med = df[sensors].median()
    scale = (df[sensors].quantile(0.75) - df[sensors].quantile(0.25)).replace(0, np.nan)
    w = 1.0 / df.groupby(BATCH_COL, sort=False)[BATCH_COL].transform("size")
    rows = (df[sensors].sample(min(20_000, len(df)), weights=w, random_state=SEED) - med) / scale
    means = (stats["mean"][sensors].loc[labels.index] - med) / scale
    pos = np.arange(1, len(sensors) + 1)
    rng = np.random.default_rng(SEED)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
    ax1.boxplot(
        [rows[c].dropna().to_numpy() for c in sensors],
        positions=pos,
        flierprops={"markersize": 1.5, "alpha": 0.3},
    )
    ax2.boxplot([means[c].dropna().to_numpy() for c in sensors], positions=pos, showfliers=False)
    for lab in (LABEL_ACCEPTED, LABEL_REJECTED):
        m = means.loc[labels.index[labels == lab]]
        for x, c in zip(pos, sensors, strict=True):
            ax2.scatter(
                x + rng.uniform(-0.25, 0.25, len(m)), m[c], s=7, alpha=0.6, color=COLORS[lab]
            )
    ax1.set_title("Rows (every batch weighted equally)")
    ax2.set_title("Batch means (one dot per batch; orange = rejected)")
    for ax in (ax1, ax2):
        ax.set_xticks(pos, sensors, rotation=90)
        ax.set_yscale("symlog", linthresh=3)
    ax1.set_ylabel("(x - median) / IQR")
    fig.tight_layout()
    plt.show()


def apply_cleaning(df, drop_columns, drop_rows=None, drop_duplicates=True):
    """Cleaned copy: columns removed, optionally the rows flagged in `drop_rows`, exact duplicate
    rows within a batch removed, and the time index t renumbered per batch afterwards."""
    out = df.drop(columns=list(drop_columns))
    if drop_rows is not None:
        out = out.loc[~drop_rows.reindex(out.index, fill_value=False)]
    if drop_duplicates:
        out = out.drop_duplicates(subset=[c for c in out.columns if c != TIME_COL])
    out = out.copy()
    out[TIME_COL] = out.groupby(BATCH_COL, sort=False).cumcount()
    return out.reset_index(drop=True)

## Process A - industrial process

Expected from the brief: labels are imbalanced and **batch length varies a lot**, so the length-aware
steps (length vs label, row weights, rescaled-time plots) matter most here.

### A.1 Load and structure
Row-order checks matter because `data.py` builds the time index from row order (rows of a batch must be contiguous).

In [ ]:
cal, labels, test = load("A")
sensors = sensor_columns(cal)
lengths, test_lengths = batch_lengths(cal), batch_lengths(test)

print(f"sensors ({len(sensors)}): {sensors}")
display(
    pd.concat(
        {"calibration": structure_checks(cal, sensors), "test": structure_checks(test, sensors)},
        axis=1,
    )
)

### A.2 Missing values
Shown pooled and batch-averaged. If they disagree, the missing cells are concentrated in short or long batches.
Long consecutive gaps (as a fraction of the batch) are worse than scattered NaNs for time-aware methods.

**Rule:** a sensor with more than 50% of its values missing is dropped (`MISSING_DROP_THRESHOLD`, decided on calibration, applied to both calibration and test). Below that, NaNs are kept for now: imputing is a modelling choice (e.g. forward-fill within a batch, never across batches).

In [ ]:
miss, miss_per_batch = missing_report(cal, sensors)
test_miss, test_per_batch = missing_report(test, sensors)
print(
    f"missing cells - calibration: {int(cal[sensors].isna().sum().sum()):,} | "
    f"test: {int(test[sensors].isna().sum().sum()):,}"
)
affected = miss.join(test_miss.add_prefix("test_"))
affected = affected[(affected["batches_affected"] > 0) | (affected["test_batches_affected"] > 0)]
if affected.empty:
    print("no missing values in any sensor (calibration or test)")
else:
    display(affected)

if miss["batches_affected"].any():
    nan_gaps = longest_run_fraction(cal, sensors, "nan")
    print("Worst batches by longest NaN gap (fraction of the batch length):")
    display(
        nan_gaps.max(axis=1).sort_values(ascending=False).head(5).to_frame("longest_nan_gap_frac")
    )

# Rule of thumb: drop a sensor when more than MISSING_DROP_THRESHOLD of its values are missing.
drop_cols = high_missing_sensors(miss_per_batch)
test_only = high_missing_sensors(test_per_batch).index.difference(drop_cols.index)
if len(test_only):
    print(f"WARNING: above the threshold in test only (kept, check manually): {list(test_only)}")
print(
    f"sensors above the {MISSING_DROP_THRESHOLD:.0%} missing threshold: "
    f"{list(drop_cols.index) or 'none'}"
)
if len(drop_cols):
    display(drop_cols.to_frame())
sensors = [s for s in sensors if s not in drop_cols.index]
n_empty = int((miss_per_batch[sensors] == 1).sum().sum())
print(f"batch x sensor pairs that are completely missing (sensors kept): {n_empty}")

### A.3 Batch length vs label
Is length related to the label (e.g. aborted runs are rejected)? `flag_short_batches` is the one rule left for you to define.

In [ ]:
len_summary, len_auc = length_vs_label(lengths, labels)
display(len_summary)
print(
    f"AUC of batch length for predicting 'rejected': {len_auc:.2f} "
    "(0.5 = unrelated, <0.5 = rejected are shorter)"
)

print("shortest batches (useful to choose the short-batch rule):")
display(lengths.sort_values().head(8).to_frame().join(labels.rename("label")))

short = flag_short_batches(lengths)
print(
    f"short batches flagged: {int(short.sum())} -> "
    f"labels: {labels.loc[short[short].index].value_counts().to_dict()}"
)
plot_length_hist(lengths, labels)

### A.4 Class imbalance and how we handle it
- Batch share vs row share: they differ when one class has shorter batches.
- `accept everything` F1 is the floor every model must beat.
- **Balanced class weights** (per batch) combined with **1/n_rows** give one weight per row, so each batch contributes
  its class weight in total, whatever its length. Oversampling synthetic batches is not used: it would invent time series.
- The fold table checks that stratified 5-fold keeps rejected batches (and batch length) comparable across folds.

In [ ]:
imb, f1_accept_all = imbalance_report(labels, lengths)
display(imb)
counts = labels.value_counts()
print(
    f"imbalance ratio {counts.max() / counts.min():.1f}:1 | "
    f"'accept everything' F1 (accepted = positive): {f1_accept_all:.3f}"
)
print(
    "balanced class weights per batch:", {k: round(v, 2) for k, v in class_weights(labels).items()}
)

w = row_weights(cal, labels)
batch_w = w.groupby(cal[BATCH_COL], sort=False).sum()
print(f"row weights sum to {w.sum():.1f} (= {len(labels)} batches); weight per batch by class:")
display(batch_w.groupby(labels).agg(["min", "max"]))

print(f"held-out composition of the {N_SPLITS} stratified folds (seed {SEED}):")
display(fold_report(labels, lengths))

### A.5 Sensor characteristics
- `n_unique` / `top_value_share_%`: constant or near-constant sensors carry no signal.
- `between/within`: how much of the variation is batch-to-batch vs inside a batch.
- Stuck sensors: long runs of identical non-zero values as a fraction of the batch (zero runs = actuator off, ignored).
- Outlier batches: robust z-score (median / MAD) of per-batch mean and std, with the sensor and statistic that drove it.
  Very large z means the normal batch-to-batch spread of that statistic is tiny, so even a small shift stands out.
- Cohen's d: which sensors differ most between rejected and accepted (indicative only).

In [ ]:
stats = batch_stats(cal, sensors)
overview = sensor_overview(cal, sensors, stats)
display(overview)

constant = overview.index[overview["n_unique"] <= 1].tolist()
near_constant = overview.index[
    (overview["top_value_share_%"] > 99) & (overview["n_unique"] > 1)
].tolist()
varying = [s for s in sensors if s not in constant]
print(f"constant sensors: {constant}")
print(f"  matches CONSTANT_COLUMNS in data.py: {sorted(constant) == sorted(CONSTANT_COLUMNS['A'])}")
print(f"near-constant (>99% one value): {near_constant}")

stuck = longest_run_fraction(cal, varying, "stuck")
display(pd.DataFrame({"max_stuck_frac": stuck.max(), "batches_>20%_stuck": (stuck > 0.2).sum()}))

bo = batch_outlier_table(stats)
print("most unusual batches (robust z of per-batch mean/std):")
display(bo.join(labels.rename("label")).sort_values("max_abs_z", ascending=False).head(8))

ro = row_outlier_share(cal, varying)
display(pd.DataFrame({"batch_avg_%": ro.mean() * 100, "worst_batch_%": ro.max() * 100}))

print("effect size per sensor, rejected vs accepted:")
display(label_effect_sizes(stats, labels).to_frame())

### A.6 Correlation between sensors
Weighted so every batch counts once. Low correlation means sensors carry mostly independent information (relevant for PCA-based monitoring).

In [ ]:
corr = weighted_corr(cal, varying)
plot_corr(corr)
iu = np.triu_indices(len(corr), 1)
pairs = (
    pd.Series(
        corr.to_numpy()[iu],
        index=pd.MultiIndex.from_arrays([corr.index[iu[0]], corr.columns[iu[1]]]),
    )
    .abs()
    .sort_values(ascending=False)
)
print(f"share of sensor pairs with |r| > 0.7: {(pairs > 0.7).mean():.0%}")
display(pairs.head(5).to_frame("abs_corr"))

### A.7 Calibration vs test
The test set has no labels, but we can check it looks like the calibration data: batch lengths, missing values, and the shift of per-batch sensor means (in units of calibration batch-to-batch SD; |shift| > 1 deserves a look).

In [ ]:
test_stats = batch_stats(test, sensors)
display(calibration_vs_test(stats, test_stats).sort_values(key=np.abs, ascending=False).to_frame())
display(pd.concat({"calibration": lengths.describe(), "test": test_lengths.describe()}, axis=1))

### A.8 Trajectories
A few batches per class with time rescaled to 0..1 (for variable-length batches this is a visual aid only; aborted runs are compressed).

In [ ]:
plot_trajectories(cal, labels, varying)

### A.9 Columns to drop
Rules: **high missingness** (more than 50%, step A.2), **zero variance / constant**, **high cardinality**
(non-numeric ID-like columns; a continuous sensor has many distinct values by nature) and **multicollinearity**.

Only the clear-cut ones are dropped automatically. Review-only (listed, not dropped): near-constant sensors
(one value in more than 95% of rows) and low-cardinality sensors (2 to 5 distinct values, switch-like, kept as a flag).

**Multicollinearity** is read from the correlation plot (step A.6); pairs with |r| above 0.95 are listed with the
sensor proposed for dropping. Not dropped by default (`DROP_COLLINEAR = False`): PCA-based monitoring *uses* the
correlation structure, and a batch can be rejected precisely because that structure breaks. Dropping one sensor of a pair
removes that check. Switch it on for models that need independent inputs (e.g. plain logistic regression).

In [ ]:
card = cardinality_report(cal, sensors)
lowvar = low_variance_report(overview)
coll = collinear_candidates(corr)

zero_var = lowvar.index[lowvar["zero_variance"]].tolist()
id_like = card.index[card["high_cardinality"]].tolist()
to_drop = list(
    dict.fromkeys(
        [*drop_cols.index, *zero_var, *id_like, *(coll["drop"] if DROP_COLLINEAR else [])]
    )
)

print(f"drop - high missingness : {list(drop_cols.index)}")
print(f"drop - zero variance    : {zero_var}")
print(f"drop - high cardinality : {id_like}")
print(f"review - near-constant  : {lowvar.index[lowvar['near_constant']].tolist()}")
low_card = card.index[card["low_cardinality"] & ~lowvar["zero_variance"]].tolist()
print(f"review - low cardinality (2 to {LOW_CARDINALITY} values): {low_card}")
print(
    f"multicollinear pairs, |r| > {COLLINEAR_R} "
    f"(dropped only if DROP_COLLINEAR; now {DROP_COLLINEAR}):"
)
display(coll if len(coll) else pd.DataFrame({"": ["none"]}))
print(f"columns dropped: {len(to_drop)} -> {to_drop}")

### A.10 Rows to drop
- **Duplicates:** exact duplicate rows inside a batch, duplicate batches, and batches that are identical in calibration
  and test (leakage).
- **Extreme outliers:** a row-level spike is a value more than 10 robust SDs from the rolling median of its *own* batch,
  so a batch that is merely shifted is not flagged (that is a batch-level anomaly, step A.5). Box plots: left the rows,
  right one dot per batch.

**Whole batches are never dropped for being outliers**: here the outlier batches are mostly the rejected ones, which is the
signal. Spike rows are not dropped by default (`DROP_SPIKE_ROWS = False`): removing rows from a time series leaves gaps in `t`,
and the table shows whether spikes differ between accepted and rejected batches. Test rows are never dropped: every test batch
needs a prediction.

In [ ]:
n_dup_rows = int(cal.duplicated(subset=[BATCH_COL, *sensors]).sum())
fp_cal, fp_test = batch_fingerprints(cal, sensors), batch_fingerprints(test, sensors)
n_dup_batches = int(fp_cal.duplicated().sum()) + int(fp_test.duplicated().sum())
n_overlap = len(set(fp_cal) & set(fp_test))
print(f"duplicate rows (calibration): {n_dup_rows}")
print(f"duplicate batches (calibration + test): {n_dup_batches}")
print(f"batches identical in calibration and test (leakage): {n_overlap}")

spikes = spike_mask(cal, varying)
spike_rows = spikes.any(axis=1)
print(f"rows with an extreme spike: {spike_rows.sum():,} ({spike_rows.mean():.2%} of rows)")
display(
    pd.DataFrame(
        {
            "rows_with_spike_%": spike_rows.groupby(cal[BATCH_COL].map(labels)).mean() * 100,
            "batches_with_spike_%": spike_rows.groupby(cal[BATCH_COL], sort=False)
            .any()
            .groupby(labels)
            .mean()
            * 100,
        }
    )
)
display((spikes.mean() * 100).sort_values(ascending=False).head(5).to_frame("rows_with_spike_%"))
plot_boxplots(cal, labels, varying, stats)

clean_cal = apply_cleaning(cal, to_drop, drop_rows=spike_rows if DROP_SPIKE_ROWS else None)
clean_test = apply_cleaning(test, to_drop, drop_duplicates=False)
print(f"calibration {cal.shape} -> {clean_cal.shape} | test {test.shape} -> {clean_test.shape}")
assert clean_cal[BATCH_COL].nunique() == len(labels), "a calibration batch was lost"
assert clean_test[BATCH_COL].nunique() == test[BATCH_COL].nunique(), "a test batch was lost"

### A.11 Summary and per-batch table

In [ ]:
batch_table = pd.DataFrame(
    {
        "label": labels,
        "n_rows": lengths,
        "missing_%": miss_per_batch.mean(axis=1) * 100,
        "max_stuck_frac": stuck.max(axis=1),
        "n_extreme_stats": bo["n_extreme_stats"],
        "worst_sensor": bo["worst_sensor"],
        "short_flag": short,
    }
).loc[lengths.index]
OUTPUTS_DIR.mkdir(exist_ok=True)
batch_table.to_csv(OUTPUTS_DIR / "data_quality_A_batches.csv")

summary["A"] = {
    "calibration batches": len(labels),
    "rejected batches": int((labels == LABEL_REJECTED).sum()),
    "imbalance ratio": counts.max() / counts.min(),
    "accept-everything F1": f1_accept_all,
    "rows per batch (min)": lengths.min(),
    "rows per batch (median)": lengths.median(),
    "rows per batch (max)": lengths.max(),
    "missing cells (cal)": int(cal[sensors].isna().sum().sum()),
    "sensors dropped (high missing)": len(drop_cols),
    "constant sensors": len(constant),
    "stuck-sensor batches": int((stuck > 0.2).any(axis=1).sum()),
    "extreme batches (|z| > 4)": int((bo["n_extreme_stats"] > 0).sum()),
    "  of which rejected": int(
        ((bo["n_extreme_stats"] > 0) & (labels.loc[bo.index] == LABEL_REJECTED)).sum()
    ),
    "length AUC for rejected": len_auc,
    "short batches flagged": int(short.sum()),
    "columns to drop": len(to_drop),
    "collinear pairs (|r| > 0.95)": len(coll),
    "duplicate rows": n_dup_rows,
    "duplicate batches": n_dup_batches,
    "calibration/test identical batches": n_overlap,
    "rows with extreme spike (%)": spike_rows.mean() * 100,
    "pairs with |r| > 0.7": (pairs > 0.7).mean(),
}
display(pd.Series(summary["A"]).to_frame("Process A"))

## Process B - simulated fed-batch fermentation

Expected: **fixed batch length**, so length weighting changes nothing (kept so both sections stay
identical), 4 constant set-point columns, and **very few rejected batches** - the fold check and the
effect sizes are the cells to read carefully.

### B.1 Load and structure
Row-order checks matter because `data.py` builds the time index from row order (rows of a batch must be contiguous).

In [ ]:
cal, labels, test = load("B")
sensors = sensor_columns(cal)
lengths, test_lengths = batch_lengths(cal), batch_lengths(test)

print(f"sensors ({len(sensors)}): {sensors}")
display(
    pd.concat(
        {"calibration": structure_checks(cal, sensors), "test": structure_checks(test, sensors)},
        axis=1,
    )
)

### B.2 Missing values
Shown pooled and batch-averaged. If they disagree, the missing cells are concentrated in short or long batches.
Long consecutive gaps (as a fraction of the batch) are worse than scattered NaNs for time-aware methods.

**Rule:** a sensor with more than 50% of its values missing is dropped (`MISSING_DROP_THRESHOLD`, decided on calibration, applied to both calibration and test). Below that, NaNs are kept for now: imputing is a modelling choice (e.g. forward-fill within a batch, never across batches).

In [ ]:
miss, miss_per_batch = missing_report(cal, sensors)
test_miss, test_per_batch = missing_report(test, sensors)
print(
    f"missing cells - calibration: {int(cal[sensors].isna().sum().sum()):,} | "
    f"test: {int(test[sensors].isna().sum().sum()):,}"
)
affected = miss.join(test_miss.add_prefix("test_"))
affected = affected[(affected["batches_affected"] > 0) | (affected["test_batches_affected"] > 0)]
if affected.empty:
    print("no missing values in any sensor (calibration or test)")
else:
    display(affected)

if miss["batches_affected"].any():
    nan_gaps = longest_run_fraction(cal, sensors, "nan")
    print("Worst batches by longest NaN gap (fraction of the batch length):")
    display(
        nan_gaps.max(axis=1).sort_values(ascending=False).head(5).to_frame("longest_nan_gap_frac")
    )

# Rule of thumb: drop a sensor when more than MISSING_DROP_THRESHOLD of its values are missing.
drop_cols = high_missing_sensors(miss_per_batch)
test_only = high_missing_sensors(test_per_batch).index.difference(drop_cols.index)
if len(test_only):
    print(f"WARNING: above the threshold in test only (kept, check manually): {list(test_only)}")
print(
    f"sensors above the {MISSING_DROP_THRESHOLD:.0%} missing threshold: "
    f"{list(drop_cols.index) or 'none'}"
)
if len(drop_cols):
    display(drop_cols.to_frame())
sensors = [s for s in sensors if s not in drop_cols.index]
n_empty = int((miss_per_batch[sensors] == 1).sum().sum())
print(f"batch x sensor pairs that are completely missing (sensors kept): {n_empty}")

### B.3 Batch length vs label
Is length related to the label (e.g. aborted runs are rejected)? `flag_short_batches` is the one rule left for you to define.

In [ ]:
len_summary, len_auc = length_vs_label(lengths, labels)
display(len_summary)
print(
    f"AUC of batch length for predicting 'rejected': {len_auc:.2f} "
    "(0.5 = unrelated, <0.5 = rejected are shorter)"
)

print("shortest batches (useful to choose the short-batch rule):")
display(lengths.sort_values().head(8).to_frame().join(labels.rename("label")))

short = flag_short_batches(lengths)
print(
    f"short batches flagged: {int(short.sum())} -> "
    f"labels: {labels.loc[short[short].index].value_counts().to_dict()}"
)
plot_length_hist(lengths, labels)

### B.4 Class imbalance and how we handle it
- Batch share vs row share: they differ when one class has shorter batches.
- `accept everything` F1 is the floor every model must beat.
- **Balanced class weights** (per batch) combined with **1/n_rows** give one weight per row, so each batch contributes
  its class weight in total, whatever its length. Oversampling synthetic batches is not used: it would invent time series.
- The fold table checks that stratified 5-fold keeps rejected batches (and batch length) comparable across folds.

In [ ]:
imb, f1_accept_all = imbalance_report(labels, lengths)
display(imb)
counts = labels.value_counts()
print(
    f"imbalance ratio {counts.max() / counts.min():.1f}:1 | "
    f"'accept everything' F1 (accepted = positive): {f1_accept_all:.3f}"
)
print(
    "balanced class weights per batch:", {k: round(v, 2) for k, v in class_weights(labels).items()}
)

w = row_weights(cal, labels)
batch_w = w.groupby(cal[BATCH_COL], sort=False).sum()
print(f"row weights sum to {w.sum():.1f} (= {len(labels)} batches); weight per batch by class:")
display(batch_w.groupby(labels).agg(["min", "max"]))

print(f"held-out composition of the {N_SPLITS} stratified folds (seed {SEED}):")
display(fold_report(labels, lengths))

### B.5 Sensor characteristics
- `n_unique` / `top_value_share_%`: constant or near-constant sensors carry no signal.
- `between/within`: how much of the variation is batch-to-batch vs inside a batch.
- Stuck sensors: long runs of identical non-zero values as a fraction of the batch (zero runs = actuator off, ignored).
- Outlier batches: robust z-score (median / MAD) of per-batch mean and std, with the sensor and statistic that drove it.
  Very large z means the normal batch-to-batch spread of that statistic is tiny, so even a small shift stands out.
- Cohen's d: which sensors differ most between rejected and accepted (indicative only).

In [ ]:
stats = batch_stats(cal, sensors)
overview = sensor_overview(cal, sensors, stats)
display(overview)

constant = overview.index[overview["n_unique"] <= 1].tolist()
near_constant = overview.index[
    (overview["top_value_share_%"] > 99) & (overview["n_unique"] > 1)
].tolist()
varying = [s for s in sensors if s not in constant]
print(f"constant sensors: {constant}")
print(f"  matches CONSTANT_COLUMNS in data.py: {sorted(constant) == sorted(CONSTANT_COLUMNS['B'])}")
print(f"near-constant (>99% one value): {near_constant}")

stuck = longest_run_fraction(cal, varying, "stuck")
display(pd.DataFrame({"max_stuck_frac": stuck.max(), "batches_>20%_stuck": (stuck > 0.2).sum()}))

bo = batch_outlier_table(stats)
print("most unusual batches (robust z of per-batch mean/std):")
display(bo.join(labels.rename("label")).sort_values("max_abs_z", ascending=False).head(8))

ro = row_outlier_share(cal, varying)
display(pd.DataFrame({"batch_avg_%": ro.mean() * 100, "worst_batch_%": ro.max() * 100}))

print("effect size per sensor, rejected vs accepted:")
display(label_effect_sizes(stats, labels).to_frame())

### B.6 Correlation between sensors
Weighted so every batch counts once. Low correlation means sensors carry mostly independent information (relevant for PCA-based monitoring).

In [ ]:
corr = weighted_corr(cal, varying)
plot_corr(corr)
iu = np.triu_indices(len(corr), 1)
pairs = (
    pd.Series(
        corr.to_numpy()[iu],
        index=pd.MultiIndex.from_arrays([corr.index[iu[0]], corr.columns[iu[1]]]),
    )
    .abs()
    .sort_values(ascending=False)
)
print(f"share of sensor pairs with |r| > 0.7: {(pairs > 0.7).mean():.0%}")
display(pairs.head(5).to_frame("abs_corr"))

### B.7 Calibration vs test
The test set has no labels, but we can check it looks like the calibration data: batch lengths, missing values, and the shift of per-batch sensor means (in units of calibration batch-to-batch SD; |shift| > 1 deserves a look).

In [ ]:
test_stats = batch_stats(test, sensors)
display(calibration_vs_test(stats, test_stats).sort_values(key=np.abs, ascending=False).to_frame())
display(pd.concat({"calibration": lengths.describe(), "test": test_lengths.describe()}, axis=1))

### B.8 Trajectories
A few batches per class with time rescaled to 0..1 (for variable-length batches this is a visual aid only; aborted runs are compressed).

In [ ]:
plot_trajectories(cal, labels, varying)

### B.9 Columns to drop
Rules: **high missingness** (more than 50%, step B.2), **zero variance / constant**, **high cardinality**
(non-numeric ID-like columns; a continuous sensor has many distinct values by nature) and **multicollinearity**.

Only the clear-cut ones are dropped automatically. Review-only (listed, not dropped): near-constant sensors
(one value in more than 95% of rows) and low-cardinality sensors (2 to 5 distinct values, switch-like, kept as a flag).

**Multicollinearity** is read from the correlation plot (step B.6); pairs with |r| above 0.95 are listed with the
sensor proposed for dropping. Not dropped by default (`DROP_COLLINEAR = False`): PCA-based monitoring *uses* the
correlation structure, and a batch can be rejected precisely because that structure breaks. Dropping one sensor of a pair
removes that check. Switch it on for models that need independent inputs (e.g. plain logistic regression).

In [ ]:
card = cardinality_report(cal, sensors)
lowvar = low_variance_report(overview)
coll = collinear_candidates(corr)

zero_var = lowvar.index[lowvar["zero_variance"]].tolist()
id_like = card.index[card["high_cardinality"]].tolist()
to_drop = list(
    dict.fromkeys(
        [*drop_cols.index, *zero_var, *id_like, *(coll["drop"] if DROP_COLLINEAR else [])]
    )
)

print(f"drop - high missingness : {list(drop_cols.index)}")
print(f"drop - zero variance    : {zero_var}")
print(f"drop - high cardinality : {id_like}")
print(f"review - near-constant  : {lowvar.index[lowvar['near_constant']].tolist()}")
low_card = card.index[card["low_cardinality"] & ~lowvar["zero_variance"]].tolist()
print(f"review - low cardinality (2 to {LOW_CARDINALITY} values): {low_card}")
print(
    f"multicollinear pairs, |r| > {COLLINEAR_R} "
    f"(dropped only if DROP_COLLINEAR; now {DROP_COLLINEAR}):"
)
display(coll if len(coll) else pd.DataFrame({"": ["none"]}))
print(f"columns dropped: {len(to_drop)} -> {to_drop}")

### B.10 Rows to drop
- **Duplicates:** exact duplicate rows inside a batch, duplicate batches, and batches that are identical in calibration
  and test (leakage).
- **Extreme outliers:** a row-level spike is a value more than 10 robust SDs from the rolling median of its *own* batch,
  so a batch that is merely shifted is not flagged (that is a batch-level anomaly, step B.5). Box plots: left the rows,
  right one dot per batch.

**Whole batches are never dropped for being outliers**: here the outlier batches are mostly the rejected ones, which is the
signal. Spike rows are not dropped by default (`DROP_SPIKE_ROWS = False`): removing rows from a time series leaves gaps in `t`,
and the table shows whether spikes differ between accepted and rejected batches. Test rows are never dropped: every test batch
needs a prediction.

In [ ]:
n_dup_rows = int(cal.duplicated(subset=[BATCH_COL, *sensors]).sum())
fp_cal, fp_test = batch_fingerprints(cal, sensors), batch_fingerprints(test, sensors)
n_dup_batches = int(fp_cal.duplicated().sum()) + int(fp_test.duplicated().sum())
n_overlap = len(set(fp_cal) & set(fp_test))
print(f"duplicate rows (calibration): {n_dup_rows}")
print(f"duplicate batches (calibration + test): {n_dup_batches}")
print(f"batches identical in calibration and test (leakage): {n_overlap}")

spikes = spike_mask(cal, varying)
spike_rows = spikes.any(axis=1)
print(f"rows with an extreme spike: {spike_rows.sum():,} ({spike_rows.mean():.2%} of rows)")
display(
    pd.DataFrame(
        {
            "rows_with_spike_%": spike_rows.groupby(cal[BATCH_COL].map(labels)).mean() * 100,
            "batches_with_spike_%": spike_rows.groupby(cal[BATCH_COL], sort=False)
            .any()
            .groupby(labels)
            .mean()
            * 100,
        }
    )
)
display((spikes.mean() * 100).sort_values(ascending=False).head(5).to_frame("rows_with_spike_%"))
plot_boxplots(cal, labels, varying, stats)

clean_cal = apply_cleaning(cal, to_drop, drop_rows=spike_rows if DROP_SPIKE_ROWS else None)
clean_test = apply_cleaning(test, to_drop, drop_duplicates=False)
print(f"calibration {cal.shape} -> {clean_cal.shape} | test {test.shape} -> {clean_test.shape}")
assert clean_cal[BATCH_COL].nunique() == len(labels), "a calibration batch was lost"
assert clean_test[BATCH_COL].nunique() == test[BATCH_COL].nunique(), "a test batch was lost"

### B.11 Summary and per-batch table

In [ ]:
batch_table = pd.DataFrame(
    {
        "label": labels,
        "n_rows": lengths,
        "missing_%": miss_per_batch.mean(axis=1) * 100,
        "max_stuck_frac": stuck.max(axis=1),
        "n_extreme_stats": bo["n_extreme_stats"],
        "worst_sensor": bo["worst_sensor"],
        "short_flag": short,
    }
).loc[lengths.index]
OUTPUTS_DIR.mkdir(exist_ok=True)
batch_table.to_csv(OUTPUTS_DIR / "data_quality_B_batches.csv")

summary["B"] = {
    "calibration batches": len(labels),
    "rejected batches": int((labels == LABEL_REJECTED).sum()),
    "imbalance ratio": counts.max() / counts.min(),
    "accept-everything F1": f1_accept_all,
    "rows per batch (min)": lengths.min(),
    "rows per batch (median)": lengths.median(),
    "rows per batch (max)": lengths.max(),
    "missing cells (cal)": int(cal[sensors].isna().sum().sum()),
    "sensors dropped (high missing)": len(drop_cols),
    "constant sensors": len(constant),
    "stuck-sensor batches": int((stuck > 0.2).any(axis=1).sum()),
    "extreme batches (|z| > 4)": int((bo["n_extreme_stats"] > 0).sum()),
    "  of which rejected": int(
        ((bo["n_extreme_stats"] > 0) & (labels.loc[bo.index] == LABEL_REJECTED)).sum()
    ),
    "length AUC for rejected": len_auc,
    "short batches flagged": int(short.sum()),
    "columns to drop": len(to_drop),
    "collinear pairs (|r| > 0.95)": len(coll),
    "duplicate rows": n_dup_rows,
    "duplicate batches": n_dup_batches,
    "calibration/test identical batches": n_overlap,
    "rows with extreme spike (%)": spike_rows.mean() * 100,
    "pairs with |r| > 0.7": (pairs > 0.7).mean(),
}
display(pd.Series(summary["B"]).to_frame("Process B"))

## A vs B at a glance

In [ ]:
display(pd.DataFrame(summary))